# Feature Importance Analysis

Analyze all 216 features to identify which to keep and which to prune.

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb

INPUT_PATH = '/kaggle/input/sumo-data-04'

# Load model and features
model = lgb.Booster(model_file=f"{INPUT_PATH}/winner_model.lgb")
feature_cols = pd.read_csv(f"{INPUT_PATH}/feature_columns.csv")['0'].tolist()
print(f"Total features: {len(feature_cols)}")

In [ ]:
# Get feature importance
importance = pd.DataFrame({
    'feature': feature_cols,
    'gain': model.feature_importance(importance_type='gain'),
    'split': model.feature_importance(importance_type='split')
})

# Add percentage of total gain
importance['gain_pct'] = importance['gain'] / importance['gain'].sum() * 100
importance['cumulative_gain_pct'] = importance.sort_values('gain', ascending=False)['gain_pct'].cumsum()

importance = importance.sort_values('gain', ascending=False).reset_index(drop=True)
importance.index = importance.index + 1  # 1-indexed rank
importance.index.name = 'rank'

In [ ]:
print("="*80)
print("ALL FEATURES RANKED BY IMPORTANCE (gain)")
print("="*80)
pd.set_option('display.max_rows', 250)
print(importance.to_string())

In [ ]:
print("\n" + "="*80)
print("ZERO IMPORTANCE FEATURES (candidates for removal)")
print("="*80)
zero_imp = importance[importance['gain'] == 0]
print(f"Count: {len(zero_imp)}\n")
for f in zero_imp['feature'].tolist():
    print(f"  - {f}")

In [ ]:
print("\n" + "="*80)
print("LOW IMPORTANCE FEATURES (gain < 100)")
print("="*80)
low_imp = importance[(importance['gain'] > 0) & (importance['gain'] < 100)]
print(f"Count: {len(low_imp)}\n")
for _, row in low_imp.iterrows():
    print(f"  - {row['feature']}: gain={row['gain']:.1f}")

In [ ]:
print("\n" + "="*80)
print("SUMMARY")
print("="*80)
print(f"Total features: {len(importance)}")
print(f"Zero importance: {len(zero_imp)} ({len(zero_imp)/len(importance)*100:.1f}%)")
print(f"Low importance (<100): {len(low_imp)} ({len(low_imp)/len(importance)*100:.1f}%)")

# How many features for 90% of gain?
top_90 = importance[importance['cumulative_gain_pct'] <= 90]
print(f"\nFeatures for 90% of predictive power: {len(top_90)}")
print(f"Features for 95% of predictive power: {len(importance[importance['cumulative_gain_pct'] <= 95])}")
print(f"Features for 99% of predictive power: {len(importance[importance['cumulative_gain_pct'] <= 99])}")

In [ ]:
# Save full importance to CSV
importance.to_csv('/kaggle/working/feature_importance_full.csv')
print("Saved to feature_importance_full.csv")